In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import sys
# os.makedirs('../plots', exist_ok=True)


In [4]:
df = pd.read_csv('../data/nyc_311_requests_jul2025_jun2026.csv')    
df.head()

,unique_key,created_date,closed_date,agency,complaint_type,descriptor,borough,incident_zip,status
0,65568225,2025-07-15T15:30:51.000,2025-07-31T14:14:57.000,DOHMH,Rodent,Rat Sighting,BROOKLYN,11219.0,Closed
1,65568932,2025-07-15T10:06:12.000,2025-07-15T10:51:29.000,NYPD,Illegal Parking,Commercial Overnight Parking,QUEENS,11692.0,Closed
2,65567550,2025-07-15T19:56:00.000,2025-07-15T21:01:00.000,DEP,Water System,Hydrant Running Full (WA4),QUEENS,11374.0,Closed
3,65567944,2025-07-15T08:24:53.000,2025-07-15T08:54:40.000,NYPD,Noise - Residential,Banging/Pounding,STATEN ISLAND,10306.0,Closed
4,65567116,2025-07-15T12:49:30.000,2025-07-22T14:28:31.000,DOT,Sidewalk Condition,Broken Sidewalk,MANHATTAN,10003.0,Closed


Step 1- Convert datet to datetime

In [5]:

df['created_date'] = pd.to_datetime(df['created_date'])
df['closed_date'] = pd.to_datetime(df['closed_date'])
df.head()

,unique_key,created_date,closed_date,agency,complaint_type,descriptor,borough,incident_zip,status
0,65568225,2025-07-15 15:30:51,2025-07-31 14:14:57,DOHMH,Rodent,Rat Sighting,BROOKLYN,11219.0,Closed
1,65568932,2025-07-15 10:06:12,2025-07-15 10:51:29,NYPD,Illegal Parking,Commercial Overnight Parking,QUEENS,11692.0,Closed
2,65567550,2025-07-15 19:56:00,2025-07-15 21:01:00,DEP,Water System,Hydrant Running Full (WA4),QUEENS,11374.0,Closed
3,65567944,2025-07-15 08:24:53,2025-07-15 08:54:40,NYPD,Noise - Residential,Banging/Pounding,STATEN ISLAND,10306.0,Closed
4,65567116,2025-07-15 12:49:30,2025-07-22 14:28:31,DOT,Sidewalk Condition,Broken Sidewalk,MANHATTAN,10003.0,Closed


Step 2 - Fix Zip codes: float -->5-char string

In [6]:
df['incident_zip'] = df['incident_zip'].astype('Int64').astype('string').str.zfill(5)
df.head()

,unique_key,created_date,closed_date,agency,complaint_type,descriptor,borough,incident_zip,status
0,65568225,2025-07-15 15:30:51,2025-07-31 14:14:57,DOHMH,Rodent,Rat Sighting,BROOKLYN,11219,Closed
1,65568932,2025-07-15 10:06:12,2025-07-15 10:51:29,NYPD,Illegal Parking,Commercial Overnight Parking,QUEENS,11692,Closed
2,65567550,2025-07-15 19:56:00,2025-07-15 21:01:00,DEP,Water System,Hydrant Running Full (WA4),QUEENS,11374,Closed
3,65567944,2025-07-15 08:24:53,2025-07-15 08:54:40,NYPD,Noise - Residential,Banging/Pounding,STATEN ISLAND,10306,Closed
4,65567116,2025-07-15 12:49:30,2025-07-22 14:28:31,DOT,Sidewalk Condition,Broken Sidewalk,MANHATTAN,10003,Closed


In [7]:
df['borough'] = df['borough'].replace('Unspecified', pd.NA)
df['descriptor'] = df['descriptor'].replace('N/A', pd.NA)
df['status'] = df['status'].replace('Unspecified', pd.NA)
df.head()   

,unique_key,created_date,closed_date,agency,complaint_type,descriptor,borough,incident_zip,status
0,65568225,2025-07-15 15:30:51,2025-07-31 14:14:57,DOHMH,Rodent,Rat Sighting,BROOKLYN,11219,Closed
1,65568932,2025-07-15 10:06:12,2025-07-15 10:51:29,NYPD,Illegal Parking,Commercial Overnight Parking,QUEENS,11692,Closed
2,65567550,2025-07-15 19:56:00,2025-07-15 21:01:00,DEP,Water System,Hydrant Running Full (WA4),QUEENS,11374,Closed
3,65567944,2025-07-15 08:24:53,2025-07-15 08:54:40,NYPD,Noise - Residential,Banging/Pounding,STATEN ISLAND,10306,Closed
4,65567116,2025-07-15 12:49:30,2025-07-22 14:28:31,DOT,Sidewalk Condition,Broken Sidewalk,MANHATTAN,10003,Closed


In [9]:
bad_dates = df[df['closed_date'] < df['created_date']]
len(bad_dates)
bad_dates
df.shape

(17612, 9)

Step 3 - Feature engineering:

In [10]:
#  feature engineering
df['resolution_hours'] = (df['closed_date'] - df['created_date']).dt.total_seconds() / 3600
df['month'] = df['created_date'].dt.to_period('M')
df['day_of_week'] = df['created_date'].dt.day_name()
df['hour'] = df['created_date'].dt.hour
df['is_closed'] = df['closed_date'].notna()
df.head(10)

,unique_key,created_date,closed_date,agency,complaint_type,descriptor,borough,incident_zip,status,resolution_hours,month,day_of_week,hour,is_closed
0,65568225,2025-07-15 15:30:51,2025-07-31 14:14:57,DOHMH,Rodent,Rat Sighting,BROOKLYN,11219,Closed,382.735000,2025-07,Tuesday,15,True
1,65568932,2025-07-15 10:06:12,2025-07-15 10:51:29,NYPD,Illegal Parking,Commercial Overnight Parking,QUEENS,11692,Closed,0.754722,2025-07,Tuesday,10,True
2,65567550,2025-07-15 19:56:00,2025-07-15 21:01:00,DEP,Water System,Hydrant Running Full (WA4),QUEENS,11374,Closed,1.083333,2025-07,Tuesday,19,True
3,65567944,2025-07-15 08:24:53,2025-07-15 08:54:40,NYPD,Noise - Residential,Banging/Pounding,STATEN ISLAND,10306,Closed,0.496389,2025-07,Tuesday,8,True
4,65567116,2025-07-15 12:49:30,2025-07-22 14:28:31,DOT,Sidewalk Condition,Broken Sidewalk,MANHATTAN,10003,Closed,169.650278,2025-07,Tuesday,12,True
5,65567042,2025-07-15 18:07:24,2025-07-16 05:23:29,NYPD,Illegal Parking,Blocked Hydrant,QUEENS,11421,Closed,11.268056,2025-07,Tuesday,18,True
6,65564084,2025-07-15 08:13:51,2025-08-06 00:00:00,DSNY,Graffiti,Graffiti,BRONX,10452,Closed,519.769167,2025-07,Tuesday,8,True
7,65568917,2025-07-15 20:56:51,2025-07-15 21:27:30,NYPD,Illegal Parking,Blocked Sidewalk,BROOKLYN,11205,Closed,0.510833,2025-07,Tuesday,20,True
8,65564336,2025-07-15 10:20:13,2025-07-15 14:18:06,NYPD,Illegal Parking,Parking Permit Improper Use,MANHATTAN,10039,Closed,3.964722,2025-07,Tuesday,10,True
9,65561670,2025-07-14 12:32:49,2025-07-14 17:30:30,NYPD,Illegal Parking,Double Parked Blocking Traffic,QUEENS,11378,Closed,4.961389,2025-07,Monday,12,True


In [11]:
df.loc[df['resolution_hours'] < 0, 'resolution_hours'] = pd.NA

In [12]:
df['resolution_hours'].isna().sum() - (~df['is_closed']).sum()

np.int64(6)

Step 4 -Saved the cleaned dataset as "Nyc_311_clean.csv"

In [ ]:
df.to_csv('../data/nyc_311_clean.csv', index=False)